|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Quantization<h1>|
|<h2>Lecture:</h2>|<h1><b>Fewer bytes per weight, and the trap that undoes it<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

# Half the bytes, half the time

Decode reads every weight to make every token. That was the whole point of
Part 1. It has an obvious result that nobody acts on before they measure it.
**Decode time is proportional to the weight bytes.** It is not proportional to
the parameter count.

Store the weights in one byte in the place of two. Decode must then be
approximately two times as fast. The speed does not come from better
arithmetic. It comes from a smaller read.

In [2]:
def quantize_int8(weight):
  """Symmetric, one scale for each output channel (row).
  -> (int8_weight, scales)."""
  scales = weight.abs().amax(dim=1).clamp(min=1e-8) / 127.0
  int8_weight = torch.round(weight / scales[:, None]).clamp(-127, 127).to(torch.int8)
  return int8_weight, scales

def dequantize_int8(int8_weight, scales):
  return int8_weight.float() * scales[:, None]

def relative_error(approximate, exact):
  """The mean absolute error, relative to the mean absolute value."""
  return ((approximate - exact).abs().mean() / exact.abs().mean()).item()

torch.manual_seed(0)
weight = torch.randn(4096, 4096, device='cuda')
int8_weight, scales = quantize_int8(weight)
bf16_bytes = weight.numel() * 2
int8_bytes = int8_weight.numel() + scales.numel() * 4
print(f'bf16 weights: {bf16_bytes/1e6:7.1f} MB')
print(f'int8 + scales:{int8_bytes/1e6:7.1f} MB   ({bf16_bytes/int8_bytes:.1f}x smaller)')
print(f'mean relative error: {relative_error(dequantize_int8(int8_weight, scales), weight):.4f}')

bf16 weights:    33.6 MB
int8 + scales:   16.8 MB   (2.0x smaller)
mean relative error: 0.0094


### Why per-channel and not per-tensor

One scale for the whole matrix means one outlier row sets the resolution for
every other row. Per-channel gives each output its own.

In [3]:
def quantize_per_tensor(weight):
  """One scale for the full tensor. -> the dequantized weight."""
  scale = weight.abs().max() / 127.0
  return torch.round(weight / scale).clamp(-127, 127) * scale

outlier_weight = weight.clone()
outlier_weight[0] *= 60.0            # one row with very large weights
per_tensor_error = relative_error(quantize_per_tensor(outlier_weight), outlier_weight)
per_channel_error = relative_error(dequantize_int8(*quantize_int8(outlier_weight)),
                                   outlier_weight)
print(f'per tensor:  {per_tensor_error:.4f} mean relative error')
print(f'per channel: {per_channel_error:.4f}   '
      f'({per_tensor_error/per_channel_error:.0f}x better)')

per tensor:  0.4838 mean relative error
per channel: 0.0093   (52x better)


# The part that is easy to get backwards

If you expand the weights again before the multiply, the quantization saves
nothing. Time all three versions.

In [4]:
inputs = torch.randn(1, 4096, device='cuda', dtype=torch.bfloat16)
bf16_weight = weight.to(torch.bfloat16)
bf16_scales = scales.to(torch.bfloat16)
bf16_ms = cudalib.bench_ms(lambda: inputs @ bf16_weight.t(), best_of=3)
unfused_ms = cudalib.bench_ms(
    lambda: inputs @ (int8_weight.to(torch.bfloat16) * bf16_scales[:, None]).t(), best_of=3)
print(f'bf16 matmul:            {bf16_ms:7.3f} ms')
print(f'dequantize, then matmul:{unfused_ms:7.3f} ms   <- SLOWER than no quantization')
print(f'\nyou wrote {bf16_bytes/1e6:.0f} MB of bf16 weights to memory,')
print('and read them back, in addition to the int8 weights.')

bf16 matmul:              0.081 ms
dequantize, then matmul:  0.384 ms   <- SLOWER than no quantization

you wrote 34 MB of bf16 weights to memory,
and read them back, in addition to the int8 weights.


That is the whole trap, and it is why stage 18b exists. You must apply the
scale to a value that is **already in a register**, at the end of the dot
product. That gives one multiply for each output, and not one for each weight:

$$y_n = \sum_k x_k \, w_{nk} s_n = s_n \sum_k x_k w_{nk}$$

The scale comes out of the sum. K multiplies become one multiply.

Stage 18b makes you write that kernel. It is 15 to 19 times faster than the
unfused version. It is 2.7 to 3.9 times faster than bf16 [cuBLAS](../../GLOSSARY.md#cublas) at batch 1.

    ./vc guide 18b

# The other big reader

Decode does not stream only the weights. It also reads the KV cache in full at
every step. On a long context the cache can be the larger of the two.

[FP8](../../GLOSSARY.md#fp8) fits the cache better than [INT8](../../GLOSSARY.md#int8). KV entries have a wide dynamic range, and
FP8 keeps an exponent. Your card is sm_89, so `float8_e4m3fn` is native.

In [5]:
# A KV cache whose heads have very different ranges.
head_ranges = torch.logspace(-2, 2, 8, device='cuda')[:, None, None]
kv = torch.randn(8, 2048, 128, device='cuda') * head_ranges
fp8_scale = kv.abs().amax() / 448.0                  # the largest e4m3 value
fp8_kv = (kv / fp8_scale).clamp(-448, 448).to(torch.float8_e4m3fn)
fp8_error = relative_error(fp8_kv.to(torch.float32) * fp8_scale, kv)
int8_scale = kv.abs().amax() / 127.0
int8_error = relative_error(torch.round(kv / int8_scale).clamp(-127, 127) * int8_scale, kv)
print('KV cache with a 10,000x spread across heads')
print(f'  fp8  e4m3 : {fp8_error:.4f} mean relative error')
print(f'  int8      : {int8_error:.4f}')
print(f'\nsame width, {int8_error/fp8_error:.0f}x the error, because int8 has no exponent')

KV cache with a 10,000x spread across heads
  fp8  e4m3 : 0.0225 mean relative error
  int8      : 0.0419

same width, 2x the error, because int8 has no exponent


Then measure perplexity before and after, on real text, because none of these
error numbers tell you whether the model got worse at its job. Stage 18 makes
that the gate.

    ./vc guide 18